## Legacy ADLS Access with Service Principal

### 1. Load Service Principal Credentials

Credentials are securely retrieved from Azure Key Vault through a Databricks secret scope.

In [0]:
import yaml

with open("../Lab_3/config.yml", "r") as file:
    config = yaml.safe_load(file)


In [0]:
storage_account = config["storage_account"]
container = config["container"]
scope = config["scope"]

client_id = dbutils.secrets.get(scope=scope, key="sp-databricks-adls-appid")
client_secret = dbutils.secrets.get(scope=scope, key="sp-databricks-adls-appkey")
tenant_id = dbutils.secrets.get(scope=scope, key="tenant-id")

storage_host = f"{storage_account}.dfs.core.windows.net"
base_path = f"abfss://{container}@{storage_host}/"
mount_point = "/mnt/kotenko_lab2"

### 2. Configure SPN Authentication

Configure OAuth authentication for direct access to the ADLS container using the Service Principal.

In [0]:
spark.conf.set(
    f"fs.azure.account.auth.type.{storage_host}",
    "OAuth"
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_host}",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_host}",
    client_id
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_host}",
    client_secret
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_host}",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

### 3. Test Direct Storage Access


In [0]:
files = dbutils.fs.ls(base_path)

display(files)

### 4. Test Legacy DBFS Mount

In [0]:
mount_configs = {
    f"fs.azure.account.auth.type.{storage_host}":
        "OAuth",

    f"fs.azure.account.oauth.provider.type.{storage_host}":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",

    f"fs.azure.account.oauth2.client.id.{storage_host}":
        client_id,

    f"fs.azure.account.oauth2.client.secret.{storage_host}":
        client_secret,

    f"fs.azure.account.oauth2.client.endpoint.{storage_host}":
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

In [0]:
try:
    dbutils.fs.mount(
        source=base_path,
        mount_point=mount_point,
        extra_configs=mount_configs
    )

except Exception as e:
    print(str(e))

### Conclusion

Service Principal credentials were read from the secret scope and used to access the ADLS container.

The legacy `dbutils.fs.mount()` method was tested, but it is restricted on the shared cluster. For new projects, Unity Catalog External Locations should be used instead.